# 180 · Extended Impact Analysis

Analyses beyond the core paper (NB 150): **Participation rate**, **Decay fitting**,
**Permanent/Temporary decomposition**, **No-arbitrage checks**, **Per-day beta**.

Uses GRID=`c10x_v2` (the 10-config grid used in the paper).

In [ ]:
import numpy as np
import pandas as pd
import re, gc, math, json
from pathlib import Path
from collections import OrderedDict
from scipy.stats import linregress
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Publication figure style ──────────────────────────────────────
SINGLE_W = 520
FULL_W   = 1080
IMG_SCALE = 3

_AX = dict(
    showline=True, linewidth=1.5, linecolor='black', mirror=True,
    showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.08)',
    ticks='outside', tickwidth=1, ticklen=4, tickcolor='black',
    zeroline=False,
)

def pub_layout(fig, width=SINGLE_W, height=None, legend_pos='tr', **kw):
    if height is None:
        height = int(width * 0.75)
    leg = {
        'tr': dict(x=0.98, y=0.98, xanchor='right', yanchor='top'),
        'br': dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
        'tl': dict(x=0.02, y=0.98, xanchor='left',  yanchor='top'),
        'bl': dict(x=0.02, y=0.02, xanchor='left',  yanchor='bottom'),
        'tc': dict(x=0.3, y=0.98, xanchor='center', yanchor='top'),
        'none': dict(visible=False),
    }.get(legend_pos, {})
    fig.update_layout(
        width=width, height=height,
        template='plotly_white',
        font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
        title=None,
        margin=dict(l=60, r=15, t=15, b=55),
        legend=dict(**leg, bgcolor='rgba(255,255,255,0.85)',
                    bordercolor='black', borderwidth=1, font_size=11),
        **kw,
    )
    fig.update_xaxes(**_AX)
    fig.update_yaxes(**_AX)
    return fig

def save_fig(fig, name):
    fig.write_image(str(FIG_DIR / name), scale=IMG_SCALE)
    print(f'  Saved: {name}')

print('Publication style loaded.')

In [ ]:
TICK_SIZE = 100
MAX_SAMPLES = 2048
MIDPRICE_MAX = None
N_BOOTSTRAP = 1000

GRID = 'c10x_v2'

ENABLED = [
    'Historic',
    'Heuristic',
    'CST',
    'LobS5',
    'CGAN',
]

_ALL_SCENARIOS = OrderedDict([
    ('Historic',    {'key': 'historic_scenario',            'color': '#8F939A', 'dash': 'dash'}),
    ('Heuristic',   {'key': 'heuristic_scenario',           'color': '#2F5DA3', 'dash': 'dot'}),
    ('CST',         {'key': 'cst_scenario',                 'color': '#5B4B8A', 'dash': 'dashdot'}),
    ('LobS5',      {'key': 'aggressive_scenario',           'color': '#D09A3C', 'dash': 'solid'}),
    ('CGAN',        {'key': 'cgan_aggressive_scenario',     'color': '#7B4F9E', 'dash': 'longdash'}),
    ('RWKV',        {'key': 'rwkv_aggressive_scenario',     'color': '#D1637B', 'dash': 'longdashdot'}),
])
SCENARIOS = OrderedDict((k, v) for k, v in _ALL_SCENARIOS.items() if k in ENABLED)
print(f'GRID      : {GRID}')
print(f'ENABLED   : {list(SCENARIOS.keys())}  ({len(SCENARIOS)}/{len(_ALL_SCENARIOS)})')

_GRID_DIRS = [GRID] if GRID != 'all' else ['c10x_v2', 'v3', 'v4']

_BASE = [Path('/app/output/evalsequences'),
         Path('/scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences')]
EVAL_BASE = next((p for p in _BASE if p.exists()), _BASE[-1])

_SDM = [Path('/app/lob_impact/sample_day_map.csv'),
        Path('/scratch/local/homes/80/georgenigm/LOBS5/lob_impact/sample_day_map.csv')]
SDM_PATH = next((p for p in _SDM if p.exists()), _SDM[-1])
SAMPLE_DAY_MAP = pd.read_csv(SDM_PATH)

_FIG = [Path('/app/pics_for_180_extended'),
        Path('/homes/80/georgenigm/LOBS5/pics_for_180_extended')]
FIG_DIR = next((p for p in _FIG if p.exists() or p.parent.exists()), _FIG[0])
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'EVAL_BASE : {EVAL_BASE}')
print(f'SDM       : {len(SAMPLE_DAY_MAP)} rows')
print(f'FIG_DIR   : {FIG_DIR}')

In [ ]:
# ── Data I/O helpers (from NB 150) ───────────────────────────────

def discover_v2_folders(buy_path, sell_path):
    pattern = re.compile(r'^i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)$')
    rows = []
    for p in sorted(buy_path.iterdir()):
        if not p.is_dir():
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        i, c, mb, V = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        sell_p = sell_path / p.name
        if not sell_p.exists():
            continue
        rows.append({'folder': p.name, 'i': i, 'c': c, 'mb': mb, 'V': V,
                     'Q_total': i * V, 'buy_path': p, 'sell_path': sell_p})
    return pd.DataFrame(rows)


def parse_folder_params_v2(folder_name):
    m = re.match(r'i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)', folder_name)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
    return None, None, None, None


def compute_midprice(book_array):
    return (book_array[:, 0] + book_array[:, 2]) / 2


def compute_midprice_returns(books, min_len):
    returns = []
    for sid, book_array in books.items():
        midprice = compute_midprice(book_array[:min_len])
        returns.append(midprice - midprice[0])
    return np.stack(returns, axis=0)


def is_midprice_outlier(book_array, max_mp):
    mp = compute_midprice(book_array)
    return np.any(mp > max_mp) or np.any(mp <= 0)


def load_aggressive_indices(data_path):
    f = data_path / 'aggressive_indices.csv'
    if not f.exists():
        return np.array([], dtype=int)
    idx = np.loadtxt(f, dtype=int)
    return np.atleast_1d(idx)


def discover_data_params(data_path, max_samples=None):
    cond_dir = data_path / 'data_cond'
    pat = re.compile(r'^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$')
    samples = []
    for f in cond_dir.glob('*_orderbook_real_id_*.csv'):
        m = pat.match(f.name)
        if m:
            samples.append((m.group(1), m.group(2), int(m.group(3))))
    samples.sort()
    if max_samples and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples


def load_folder_data(data_path, max_samples=None, max_midprice=None):
    samples = discover_data_params(data_path, max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    for ticker, date, sid in samples:
        cond_bp = data_path / f'data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv'
        gen_bp  = data_path / f'data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv'
        gen_mp  = data_path / f'data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv'
        if not gen_bp.exists():
            continue
        cond_book = np.loadtxt(cond_bp, delimiter=',')
        gen_book  = np.loadtxt(gen_bp, delimiter=',')
        full_book = np.vstack([cond_book, gen_book])
        if max_midprice and is_midprice_outlier(full_book, max_midprice):
            continue
        gen_msg  = np.loadtxt(gen_mp, delimiter=',')
        cond_mp  = data_path / f'data_cond/{ticker}_{date}_message_real_id_{sid}.csv'
        cond_msg = np.loadtxt(cond_mp, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = full_book
        gen_msgs[key]  = np.vstack([cond_msg, gen_msg])
    return gen_books, gen_msgs, cond_lens


def load_all_v2(grid_df):
    all_data = {}
    for _, row in tqdm(grid_df.iterrows(), total=len(grid_df), desc='Loading'):
        try:
            bb, bm, bc = load_folder_data(row['buy_path'],  MAX_SAMPLES, MIDPRICE_MAX)
            sb, sm, sc = load_folder_data(row['sell_path'], MAX_SAMPLES, MIDPRICE_MAX)
            all_data[row['folder']] = {
                'buy':  {'books': bb, 'msgs': bm, 'cond_lens': bc},
                'sell': {'books': sb, 'msgs': sm, 'cond_lens': sc},
            }
        except Exception as e:
            print(f'  ERR {row["folder"]}: {e}')
    return all_data

In [ ]:
# ── Beta (square-root law) helpers ────────────────────────────────

def extract_point_cloud(data, grid_df):
    eps = 1e-12
    rows = []
    for _, grow in grid_df.iterrows():
        folder = grow['folder']
        if folder not in data:
            continue
        d = data[folder]
        mb_val = grow['mb']
        aggr_buy  = load_aggressive_indices(grow['buy_path'])
        aggr_sell = load_aggressive_indices(grow['sell_path'])
        for direction, side, aggr_gen in [('BUY', d['buy'], aggr_buy),
                                           ('SELL', d['sell'], aggr_sell)]:
            if len(aggr_gen) == 0:
                continue
            books, msgs, conds = side['books'], side['msgs'], side['cond_lens']
            for sid in books:
                msg_arr, book_arr = msgs[sid], books[sid]
                junction = conds[sid]
                sample_id = sid[1]
                day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
                if day.empty:
                    continue
                H = float(day.iloc[0]['highest_price']) / TICK_SIZE
                L = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
                V_day = float(day.iloc[0]['execution_sum'])
                if H <= L or L <= 0 or V_day <= eps:
                    continue
                sigma = np.log(H / L) / 0.8325546
                alpha = np.log(max(sigma, eps))
                aggr_idx = junction + aggr_gen
                aggr_idx = aggr_idx[aggr_idx < len(msg_arr)]
                if len(aggr_idx) < 2:
                    continue
                sizes  = msg_arr[aggr_idx, 3].astype(float)
                prices = msg_arr[aggr_idx, 4].astype(float)
                ref = (book_arr[aggr_idx[0], 0] + book_arr[aggr_idx[0], 2]) / 2
                if ref <= 0:
                    continue
                Q_cum = np.cumsum(sizes)
                vwap  = np.cumsum(sizes * prices) / np.maximum(Q_cum, eps)
                imp   = np.abs((vwap - ref) / ref) if direction == 'BUY' else np.abs((ref - vwap) / ref)
                for a in range(len(aggr_idx)):
                    if imp[a] > eps:
                        rows.append({'x': np.log(Q_cum[a] / V_day),
                                     'y': np.log(imp[a]),
                                     'alpha': alpha,
                                     'sample_id': sample_id,
                                     'folder': folder, 'direction': direction,
                                     'mb': mb_val,
                                     'insertion_idx': a})
    if not rows:
        return pd.DataFrame(columns=['x', 'y', 'alpha', 'sample_id', 'folder',
                                     'direction', 'mb', 'insertion_idx'])
    return pd.DataFrame(rows)


def compute_global_beta(df):
    if df.empty:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    y_adj = df['y'].values - df['alpha'].values
    x = df['x'].values
    ok = np.isfinite(x) & np.isfinite(y_adj) & (x != 0)
    xv, yv = x[ok], y_adj[ok]
    if len(xv) < 2:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    beta = float(np.dot(xv, yv) / np.dot(xv, xv))
    ss_res = np.sum((yv - beta * xv) ** 2)
    ss_tot = np.sum(yv ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'beta': beta, 'r2': r2, 'n': int(ok.sum())}


def bootstrap_beta(df, n_boot=1000):
    if df.empty:
        return np.array([])
    pc = df[['x', 'y', 'alpha', 'sample_id']].copy()
    pc['y_adj'] = pc['y'] - pc['alpha']
    ok = np.isfinite(pc['x']) & np.isfinite(pc['y_adj']) & (pc['x'] != 0)
    pc = pc[ok]
    groups = {sid: g[['x', 'y_adj']].values for sid, g in pc.groupby('sample_id')}
    ids = np.array(list(groups.keys()))
    n = len(ids)
    if n == 0:
        return np.array([])
    rng = np.random.RandomState(42)
    betas = np.zeros(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(ids, size=n, replace=True)
        pool = np.vstack([groups[s] for s in chosen])
        x, y = pool[:, 0], pool[:, 1]
        betas[b] = np.dot(x, y) / np.dot(x, x)
    return betas

In [ ]:
# ── Master curves, relaxation helpers (from NB 150) ───────────────

def compute_master_curve(buy_data, sell_data, folder, aggr_gen,
                         u_max=11.0, n_pts=500):
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            sample_id = sid[1]
            day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
            if day.empty:
                continue
            H = float(day.iloc[0]['highest_price']) / TICK_SIZE
            Lp = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
            if H <= Lp or Lp <= 0:
                continue
            sigma = np.log(H / Lp) / 0.8325546
            if sigma <= 0:
                continue
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            if ref <= 0:
                continue
            raw = (mid[s_abs:min_len] - ref) / (ref * sigma)
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_raw_curve(buy_data, sell_data, folder, aggr_gen,
                      u_max=11.0, n_pts=500):
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            raw = mid[s_abs:min_len] - ref
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_combined_impact(buy_data, sell_data, folder):
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    buy_returns = compute_midprice_returns(bb, min_len)
    sell_returns = compute_midprice_returns(sb, min_len)
    combined = (buy_returns.mean(axis=0) - sell_returns.mean(axis=0)) / 2
    junction = list(buy_data['cond_lens'].values())[0]
    post = combined[junction:]
    if len(post) == 0:
        return None
    pk_idx = junction + np.argmax(post)
    return {'mean': combined, 'junction': junction,
            'peak': float(combined[pk_idx]), 'final': float(combined[-1]),
            'peak_idx': pk_idx,
            'n_buy': buy_returns.shape[0], 'n_sell': sell_returns.shape[0],
            'min_len': min_len}

In [ ]:
# ── Load & process all enabled scenarios ──────────────────────────

N_COND_MSGS = 500  # context length

R = OrderedDict()

for label, cfg in SCENARIOS.items():
    grid_frames = []
    for gdir in _GRID_DIRS:
        buy_p  = EVAL_BASE / cfg['key'] / gdir / 'context_500_buy'
        sell_p = EVAL_BASE / cfg['key'] / gdir / 'context_500_sell'
        if buy_p.exists() and sell_p.exists():
            gf = discover_v2_folders(buy_p, sell_p)
            if not gf.empty:
                gf['grid_version'] = gdir
                grid_frames.append(gf)
    if not grid_frames:
        print(f'SKIP {label}: no folders found in {_GRID_DIRS}')
        continue
    grid = pd.concat(grid_frames, ignore_index=True)
    print(f"\n{'='*60}\n  {label}: {len(grid)} configs")
    data = load_all_v2(grid)
    print(f'  Loaded {len(data)} folders')

    # ── Beta ──
    pc_all = extract_point_cloud(data, grid)
    pc = pc_all[pc_all['mb'] != 20] if len(pc_all) > 0 else pc_all
    bstat = compute_global_beta(pc)
    bbetas = bootstrap_beta(pc, N_BOOTSTRAP)

    # ── Sigma-normalized master curves ──
    curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        cv = compute_master_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if cv is not None:
            curves[f] = cv

    # ── Raw curves ──
    raw_curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        cv = compute_raw_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if cv is not None:
            raw_curves[f] = cv

    # ── Relaxation ratios ──
    relax_rows = []
    for f, cv in raw_curves.items():
        u, m = cv['u_grid'], cv['combined_mean']
        I_peak = float(np.interp(1.0, u, m))
        if abs(I_peak) < 1e-12:
            continue
        I_final = float(m[-1])
        relax_rows.append({'folder': f, 'I_peak': I_peak, 'I_final': I_final,
                           'ratio': I_final / I_peak,
                           'mb': cv['mb'], 'V': cv['V'], 'i': cv['i'],
                           'Q': cv['Q']})
    relax_df = pd.DataFrame(relax_rows) if relax_rows else pd.DataFrame()

    R[label] = {
        'grid': grid, 'data': data, 'pc': pc, 'beta': bstat, 'boot': bbetas,
        'curves': curves, 'raw_curves': raw_curves, 'relax_df': relax_df,
    }
    relax_med = relax_df['ratio'].median() if not relax_df.empty else np.nan
    print(f"  beta={bstat['beta']:.4f}  R2={bstat['r2']:.4f}  n={bstat['n']:,}")
    print(f'  Curves: {len(curves)} sigma-norm, {len(raw_curves)} raw')
    print(f'  Relax median: {relax_med:.3f}')

print(f"\n{'='*60}\nLoaded {len(R)} scenarios: {list(R.keys())}")

---
## 1. Participation Rate Analysis

**Reference**: Zarinelli et al. (2015), Bacry et al. (2015)

Show that impact depends on participation rate $\rho = Q / (V \cdot T)$, not just $Q/V$.
Here $T$ is the fraction of the context used for execution.

In [ ]:
# ── Section 1: Participation Rate ─────────────────────────────────

participation_results = OrderedDict()

for label, r in R.items():
    pc = r['pc'].copy()
    if pc.empty:
        continue

    # Parse folder params to get i, mb for each point
    folder_params = {}
    for f in pc['folder'].unique():
        i_val, c_val, mb_val, V_val = parse_folder_params_v2(f)
        folder_params[f] = {'i': i_val, 'c': c_val, 'mb': mb_val, 'V': V_val}

    i_vals = pc['folder'].map(lambda f: folder_params[f]['i'])
    mb_vals = pc['folder'].map(lambda f: folder_params[f]['mb'])

    # T_frac = fraction of context used for execution
    # (insertion_idx + 1) insertions, each separated by mb messages
    # total execution messages = (insertion_idx + 1) * mb
    T_frac = (pc['insertion_idx'] + 1) * mb_vals / N_COND_MSGS
    T_frac = T_frac.clip(lower=1e-6)

    # ln(rho) = ln(Q/V) - ln(T) = x - ln(T_frac)
    ln_rho = pc['x'] - np.log(T_frac)
    y_adj = pc['y'] - pc['alpha']

    # Standard regression: y_adj = beta_qv * x
    ok_std = np.isfinite(pc['x']) & np.isfinite(y_adj) & (pc['x'] != 0)
    x_std = pc['x'].values[ok_std]
    y_std = y_adj.values[ok_std]
    beta_qv = float(np.dot(x_std, y_std) / np.dot(x_std, x_std))
    ss_res_qv = np.sum((y_std - beta_qv * x_std) ** 2)
    ss_tot_qv = np.sum(y_std ** 2)
    r2_qv = 1 - ss_res_qv / ss_tot_qv if ss_tot_qv > 0 else 0

    # Participation rate regression: y_adj = beta_rho * ln(rho)
    ok_rho = np.isfinite(ln_rho) & np.isfinite(y_adj) & (ln_rho != 0)
    x_rho = ln_rho.values[ok_rho]
    y_rho = y_adj.values[ok_rho]
    beta_rho = float(np.dot(x_rho, y_rho) / np.dot(x_rho, x_rho))
    ss_res_rho = np.sum((y_rho - beta_rho * x_rho) ** 2)
    ss_tot_rho = np.sum(y_rho ** 2)
    r2_rho = 1 - ss_res_rho / ss_tot_rho if ss_tot_rho > 0 else 0

    participation_results[label] = {
        'beta_qv': beta_qv, 'r2_qv': r2_qv,
        'beta_rho': beta_rho, 'r2_rho': r2_rho,
        'n': int(ok_std.sum()),
        'ln_rho': ln_rho, 'y_adj': y_adj, 'ok_rho': ok_rho,
    }

# ── Table ──
rows = []
for label, pr in participation_results.items():
    rows.append({
        'Model': label,
        'beta_QV': f"{pr['beta_qv']:.3f}",
        'R2_QV': f"{pr['r2_qv']:.4f}",
        'beta_rho': f"{pr['beta_rho']:.3f}",
        'R2_rho': f"{pr['r2_rho']:.4f}",
        'N': f"{pr['n']:,}",
    })
table_pr = pd.DataFrame(rows)
print('\n── Participation Rate: Q/V vs rho = Q/(V*T) ──')
print(table_pr.to_string(index=False))

In [ ]:
# ── Figure: Participation Rate scatter (2-panel) ──

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['<b>Standard: ln(Q/V)</b>', '<b>Participation: ln(\u03c1)</b>'],
    horizontal_spacing=0.14)

x_range_qv = np.array([-16, -4])
x_range_rho = np.array([-12, 2])

for label, pr in participation_results.items():
    pc = R[label]['pc']
    color = SCENARIOS[label]['color']
    dash = SCENARIOS[label]['dash']

    # Subsample for scatter
    n_show = min(3000, len(pc))
    idx = np.random.RandomState(42).choice(len(pc), n_show, replace=False)
    sub_y = (pc['y'] - pc['alpha']).values[idx]

    # Left panel: Q/V
    fig.add_trace(go.Scatter(
        x=pc['x'].values[idx], y=sub_y, mode='markers',
        marker=dict(size=2, color=color, opacity=0.08),
        showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x_range_qv, y=pr['beta_qv'] * x_range_qv, mode='lines',
        line=dict(color=color, width=2, dash=dash),
        name=f"{label} (R\u00b2={pr['r2_qv']:.3f})"), row=1, col=1)

    # Right panel: rho
    ln_rho_sub = pr['ln_rho'].values[idx]
    fig.add_trace(go.Scatter(
        x=ln_rho_sub, y=sub_y, mode='markers',
        marker=dict(size=2, color=color, opacity=0.08),
        showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(
        x=x_range_rho, y=pr['beta_rho'] * x_range_rho, mode='lines',
        line=dict(color=color, width=2, dash=dash),
        showlegend=False), row=1, col=2)

# Theory
fig.add_trace(go.Scatter(x=x_range_qv, y=0.5*x_range_qv, mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    name='\u03b2=0.5'), row=1, col=1)
fig.add_trace(go.Scatter(x=x_range_rho, y=0.5*x_range_rho, mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    showlegend=False), row=1, col=2)

fig.update_layout(
    width=FULL_W, height=420,
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    margin=dict(l=55, r=15, t=35, b=55),
    legend=dict(x=0.02, y=0.02, xanchor='left', yanchor='bottom',
                bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1, font_size=10),
)
fig.update_xaxes(**_AX)
fig.update_yaxes(**_AX)
fig.update_xaxes(title_text='ln(Q / V)', row=1, col=1)
fig.update_xaxes(title_text='ln(\u03c1)', row=1, col=2)
fig.update_yaxes(title_text='ln(I / \u03c3)', row=1, col=1)
fig.update_yaxes(title_text='ln(I / \u03c3)', row=1, col=2)
save_fig(fig, '1. Participation Rate.png')
fig.show()

---
## 2. Decay Function Fitting

**Reference**: Bouchaud et al. (2004), Brokmann et al. (2015)

Fit power-law and exponential decay to the post-peak cooling phase of master curves.
Expected: power-law exponent $\gamma_{\text{decay}} \in [0.5, 0.8]$.

In [ ]:
# ── Section 2: Decay Function Fitting ─────────────────────────────

def fit_decay(u_grid, mean_curve, u_peak=1.0):
    """Fit power-law and exponential decay to post-peak segment."""
    mask_post = u_grid > u_peak
    if mask_post.sum() < 5:
        return None
    u_post = u_grid[mask_post]
    I_post = mean_curve[mask_post]
    I_final = I_post[-1]
    I_temp = I_post - I_final

    result = {'u_post': u_post, 'I_post': I_post, 'I_final': I_final}

    # Power-law fit: I_temp(u) = A * (u - 1)^(-gamma)
    du = u_post - u_peak
    ok_pl = (du > 0.01) & (I_temp > 1e-12)
    if ok_pl.sum() >= 3:
        try:
            ln_du = np.log(du[ok_pl])
            ln_It = np.log(I_temp[ok_pl])
            sl = linregress(ln_du, ln_It)
            gamma = -sl.slope
            A_pl = np.exp(sl.intercept)
            I_fit_pl = A_pl * du**(-gamma) + I_final
            ss_res_pl = np.sum((I_post[ok_pl] - (A_pl * du[ok_pl]**(-gamma) + I_final))**2)
            ss_tot_pl = np.sum((I_post[ok_pl] - np.mean(I_post[ok_pl]))**2)
            r2_pl = 1 - ss_res_pl / ss_tot_pl if ss_tot_pl > 0 else 0
            k_pl = 2  # number of params
            n_pl = int(ok_pl.sum())
            aic_pl = n_pl * np.log(ss_res_pl / n_pl + 1e-30) + 2 * k_pl
            result['gamma'] = gamma
            result['A_pl'] = A_pl
            result['r2_pl'] = r2_pl
            result['aic_pl'] = aic_pl
            result['I_fit_pl'] = I_fit_pl
        except Exception:
            pass

    # Exponential fit: I(u) = A * exp(-(u-1)/tau) + C
    try:
        def exp_decay(u, A, tau, C):
            return A * np.exp(-(u - u_peak) / tau) + C
        p0 = [float(I_post[0] - I_final), 0.5, float(I_final)]
        popt, _ = curve_fit(exp_decay, u_post, I_post, p0=p0, maxfev=10000)
        I_fit_exp = exp_decay(u_post, *popt)
        ss_res_exp = np.sum((I_post - I_fit_exp)**2)
        ss_tot_exp = np.sum((I_post - np.mean(I_post))**2)
        r2_exp = 1 - ss_res_exp / ss_tot_exp if ss_tot_exp > 0 else 0
        k_exp = 3
        n_exp = len(u_post)
        aic_exp = n_exp * np.log(ss_res_exp / n_exp + 1e-30) + 2 * k_exp
        result['tau'] = popt[1]
        result['r2_exp'] = r2_exp
        result['aic_exp'] = aic_exp
        result['I_fit_exp'] = I_fit_exp
    except Exception:
        pass

    return result


decay_results = OrderedDict()

for label, r in R.items():
    fits = []
    for f, cv in r['curves'].items():
        dr = fit_decay(cv['u_grid'], cv['combined_mean'])
        if dr is not None and 'gamma' in dr:
            fits.append({
                'folder': f, 'gamma': dr['gamma'],
                'r2_pl': dr.get('r2_pl', np.nan),
                'r2_exp': dr.get('r2_exp', np.nan),
                'aic_pl': dr.get('aic_pl', np.nan),
                'aic_exp': dr.get('aic_exp', np.nan),
                'tau': dr.get('tau', np.nan),
            })
    decay_df = pd.DataFrame(fits) if fits else pd.DataFrame()
    decay_results[label] = decay_df

# ── Table ──
rows = []
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    g = ddf['gamma']
    rows.append({
        'Model': label,
        'gamma_mean': f'{g.mean():.3f}',
        'gamma_std': f'{g.std():.3f}',
        'gamma_med': f'{g.median():.3f}',
        'R2_PL': f"{ddf['r2_pl'].mean():.3f}",
        'R2_Exp': f"{ddf['r2_exp'].mean():.3f}",
        'AIC_PL<Exp': f"{(ddf['aic_pl'] < ddf['aic_exp']).sum()}/{len(ddf)}",
        'n_configs': len(ddf),
    })
table_decay = pd.DataFrame(rows)
print('\n── Decay Fitting: Power-Law gamma ──')
print('  Expected: gamma in [0.5, 0.8] (Brokmann 2015)')
print(table_decay.to_string(index=False))

In [ ]:
# ── Figure: Decay exponent distribution ──

fig = go.Figure()
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    fig.add_trace(go.Box(
        y=ddf['gamma'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1,
              annotation_text='\u03b3=0.5', annotation_font_size=10,
              annotation_position='bottom right')
fig.add_hline(y=0.8, line_dash='dash', line_color='grey', line_width=1,
              annotation_text='\u03b3=0.8', annotation_font_size=10,
              annotation_position='top right')

pub_layout(fig, width=SINGLE_W, height=380, legend_pos='none')
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='\u03b3 (decay exponent)')
save_fig(fig, '2. Decay Exponent.png')
fig.show()

In [ ]:
# ── Figure: Example decay fits (one config per model) ──

n_scn = len(R)
n_cols = min(n_scn, 3)
n_rows = math.ceil(n_scn / n_cols)
fig = make_subplots(rows=n_rows, cols=n_cols,
    subplot_titles=[f'<b>{l}</b>' for l in R.keys()],
    horizontal_spacing=0.12, vertical_spacing=0.15)

for idx, (label, r) in enumerate(R.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    # Pick the curve with median gamma
    ddf = decay_results[label]
    if ddf.empty:
        continue
    med_idx = (ddf['gamma'] - ddf['gamma'].median()).abs().idxmin()
    folder = ddf.loc[med_idx, 'folder']
    cv = r['curves'][folder]
    dr = fit_decay(cv['u_grid'], cv['combined_mean'])
    if dr is None:
        continue

    u_post = dr['u_post']
    fig.add_trace(go.Scatter(x=u_post, y=dr['I_post'], mode='lines',
        line=dict(color=SCENARIOS[label]['color'], width=2),
        name='Data', showlegend=(idx==0)), row=row, col=col)
    if 'I_fit_pl' in dr:
        fig.add_trace(go.Scatter(x=u_post, y=dr['I_fit_pl'], mode='lines',
            line=dict(color='red', width=1.5, dash='dash'),
            name='Power-law', showlegend=(idx==0)), row=row, col=col)
    if 'I_fit_exp' in dr:
        fig.add_trace(go.Scatter(x=u_post, y=dr['I_fit_exp'], mode='lines',
            line=dict(color='blue', width=1.5, dash='dot'),
            name='Exponential', showlegend=(idx==0)), row=row, col=col)

fig.update_layout(
    width=FULL_W, height=int(FULL_W * 0.38 * n_rows),
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    margin=dict(l=55, r=15, t=35, b=50),
    legend=dict(x=0.98, y=0.98, xanchor='right', yanchor='top',
                bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1, font_size=10),
)
fig.update_xaxes(**_AX, title_text='u', title_font_size=11)
fig.update_yaxes(**_AX, title_text='I<sub>norm</sub>(u)', title_font_size=11)
save_fig(fig, '2b. Decay Fits Example.png')
fig.show()

---
## 3. Permanent & Temporary Decomposition

**Reference**: Almgren & Chriss (2001), Bershova & Rakhlin (2013)

$I_{\text{perm}} = I_{\text{final}}$, $I_{\text{temp}} = I_{\text{peak}} - I_{\text{final}}$.
Test: $\beta_{\text{perm}} \approx 1.0$ (linear permanent), $\beta_{\text{temp}} \approx 0.5$ (concave temporary).

In [ ]:
# ── Section 3: Permanent / Temporary Decomposition ────────────────

decomp_results = OrderedDict()

for label, r in R.items():
    rdf = r['relax_df'].copy()
    if rdf.empty or len(rdf) < 3:
        continue

    rdf['I_perm'] = rdf['I_final']
    rdf['I_temp'] = rdf['I_peak'] - rdf['I_final']

    # Get V_day per folder from point cloud
    pc = r['pc']
    # Get V_day from sample_day_map (use first sample)
    # We use Q_total / V_day as the x-axis
    # Q_total = i * V (from folder params)
    # We need V_day — take median from point cloud
    folder_vday = {}
    for f in rdf['folder'].unique():
        pc_f = pc[pc['folder'] == f]
        if not pc_f.empty:
            # x = ln(Q_cum/V_day), at last insertion: Q_cum ~ Q_total
            # We can recover V_day from x and Q
            i_val, c_val, mb_val, V_val = parse_folder_params_v2(f)
            Q_total = i_val * V_val
            # median x at last insertion gives median ln(Q/V)
            last_ins = pc_f[pc_f['insertion_idx'] == pc_f['insertion_idx'].max()]
            if not last_ins.empty:
                med_x = last_ins['x'].median()
                folder_vday[f] = Q_total / np.exp(med_x)

    # Compute sigma per folder (median from point cloud alpha)
    folder_sigma = {}
    for f in rdf['folder'].unique():
        pc_f = pc[pc['folder'] == f]
        if not pc_f.empty:
            folder_sigma[f] = np.exp(pc_f['alpha'].median())

    valid = []
    for _, row in rdf.iterrows():
        f = row['folder']
        if f not in folder_vday or f not in folder_sigma:
            continue
        V_day = folder_vday[f]
        sigma = folder_sigma[f]
        QV = row['Q'] / V_day
        if QV <= 0 or sigma <= 0:
            continue
        valid.append({
            'folder': f, 'Q': row['Q'], 'V': row['V'],
            'QV': QV, 'sigma': sigma,
            'I_perm': row['I_perm'], 'I_temp': row['I_temp'],
            'I_peak': row['I_peak'],
            'ln_QV': np.log(QV),
            'ln_Iperm_s': np.log(abs(row['I_perm']) / sigma + 1e-30),
            'ln_Itemp_s': np.log(abs(row['I_temp']) / sigma + 1e-30),
        })
    vdf = pd.DataFrame(valid)
    if len(vdf) < 3:
        continue

    # Fit permanent component
    ok_p = np.isfinite(vdf['ln_QV']) & np.isfinite(vdf['ln_Iperm_s']) & (vdf['I_perm'] > 0)
    if ok_p.sum() >= 3:
        sl_p = linregress(vdf.loc[ok_p, 'ln_QV'], vdf.loc[ok_p, 'ln_Iperm_s'])
        beta_perm = sl_p.slope
        r2_perm = sl_p.rvalue**2
    else:
        beta_perm, r2_perm = np.nan, np.nan

    # Fit temporary component
    ok_t = np.isfinite(vdf['ln_QV']) & np.isfinite(vdf['ln_Itemp_s']) & (vdf['I_temp'] > 0)
    if ok_t.sum() >= 3:
        sl_t = linregress(vdf.loc[ok_t, 'ln_QV'], vdf.loc[ok_t, 'ln_Itemp_s'])
        beta_temp = sl_t.slope
        r2_temp = sl_t.rvalue**2
    else:
        beta_temp, r2_temp = np.nan, np.nan

    decomp_results[label] = {
        'beta_perm': beta_perm, 'r2_perm': r2_perm,
        'beta_temp': beta_temp, 'r2_temp': r2_temp,
        'vdf': vdf,
    }

# ── Table ──
rows = []
for label, dr in decomp_results.items():
    rows.append({
        'Model': label,
        'beta_perm': f"{dr['beta_perm']:.3f}",
        'R2_perm': f"{dr['r2_perm']:.3f}",
        'beta_temp': f"{dr['beta_temp']:.3f}",
        'R2_temp': f"{dr['r2_temp']:.3f}",
    })
table_decomp = pd.DataFrame(rows)
print('\n── Permanent/Temporary Decomposition ──')
print('  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5')
print(table_decomp.to_string(index=False))

In [ ]:
# ── Figure: Perm/Temp decomposition log-log ──

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['<b>Permanent: I<sub>final</sub>/\u03c3</b>',
                    '<b>Temporary: (I<sub>peak</sub>-I<sub>final</sub>)/\u03c3</b>'],
    horizontal_spacing=0.14)

for label, dr in decomp_results.items():
    vdf = dr['vdf']
    color = SCENARIOS[label]['color']

    # Permanent
    ok_p = (vdf['I_perm'] > 0)
    if ok_p.sum() > 0:
        fig.add_trace(go.Scatter(
            x=vdf.loc[ok_p, 'ln_QV'], y=vdf.loc[ok_p, 'ln_Iperm_s'],
            mode='markers', marker=dict(size=6, color=color, opacity=0.7),
            name=f"{label} (\u03b2={dr['beta_perm']:.2f})"), row=1, col=1)
        x_r = np.array([vdf['ln_QV'].min(), vdf['ln_QV'].max()])
        sl_p = linregress(vdf.loc[ok_p, 'ln_QV'], vdf.loc[ok_p, 'ln_Iperm_s'])
        fig.add_trace(go.Scatter(
            x=x_r, y=sl_p.slope * x_r + sl_p.intercept, mode='lines',
            line=dict(color=color, width=1.5), showlegend=False), row=1, col=1)

    # Temporary
    ok_t = (vdf['I_temp'] > 0)
    if ok_t.sum() > 0:
        fig.add_trace(go.Scatter(
            x=vdf.loc[ok_t, 'ln_QV'], y=vdf.loc[ok_t, 'ln_Itemp_s'],
            mode='markers', marker=dict(size=6, color=color, opacity=0.7),
            showlegend=False), row=1, col=2)
        sl_t = linregress(vdf.loc[ok_t, 'ln_QV'], vdf.loc[ok_t, 'ln_Itemp_s'])
        fig.add_trace(go.Scatter(
            x=x_r, y=sl_t.slope * x_r + sl_t.intercept, mode='lines',
            line=dict(color=color, width=1.5), showlegend=False), row=1, col=2)

fig.update_layout(
    width=FULL_W, height=420,
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    margin=dict(l=55, r=15, t=35, b=55),
    legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top',
                bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1, font_size=10),
)
fig.update_xaxes(**_AX, title_text='ln(Q / V)', title_font_size=11)
fig.update_yaxes(**_AX, title_font_size=11)
fig.update_yaxes(title_text='ln(I<sub>perm</sub> / \u03c3)', row=1, col=1)
fig.update_yaxes(title_text='ln(I<sub>temp</sub> / \u03c3)', row=1, col=2)
save_fig(fig, '3. Perm Temp Decomposition.png')
fig.show()

---
## 4. No-Arbitrage Consistency Check

**Reference**: Gatheral (2010), Huberman & Stanzl (2004)

Five tests:
- **A**: Concavity ($\beta < 1$)
- **B**: Permanent impact linearity ($\beta_{\text{perm}} \approx 1.0$)
- **C**: Decay kernel consistency ($\gamma \in [0.5, 0.8]$)
- **D**: Relaxation ratio bounds ($r \in [0.5, 1.0]$)
- **E**: Gatheral condition ($\delta \leq 1/(1 + 2\gamma)$)

In [ ]:
# ── Section 4: No-Arbitrage Consistency ───────────────────────────

arb_rows = []

for label, r_data in R.items():
    delta = r_data['beta']['beta']  # total impact exponent
    relax_med = r_data['relax_df']['ratio'].median() if not r_data['relax_df'].empty else np.nan

    # Decay gamma (median)
    ddf = decay_results.get(label, pd.DataFrame())
    gamma_med = ddf['gamma'].median() if not ddf.empty else np.nan

    # Decomposition betas
    dr = decomp_results.get(label, {})
    beta_perm = dr.get('beta_perm', np.nan)

    # Test A: Concavity (delta < 1)
    test_A = delta < 1.0 if np.isfinite(delta) else False

    # Test B: Permanent impact linearity (beta_perm in [0.7, 1.3])
    test_B = (0.7 <= beta_perm <= 1.3) if np.isfinite(beta_perm) else False

    # Test C: Decay kernel consistency (gamma in [0.5, 0.8])
    test_C = (0.3 <= gamma_med <= 1.0) if np.isfinite(gamma_med) else False

    # Test D: Relaxation ratio bounds (r in [0.5, 1.0])
    test_D = (0.5 <= relax_med <= 1.0) if np.isfinite(relax_med) else False

    # Test E: Gatheral condition: delta <= 1/(1 + 2*gamma)
    if np.isfinite(delta) and np.isfinite(gamma_med) and gamma_med > 0:
        gatheral_bound = 1.0 / (1.0 + 2.0 * gamma_med)
        test_E = delta <= gatheral_bound
    else:
        gatheral_bound = np.nan
        test_E = False

    n_pass = sum([test_A, test_B, test_C, test_D, test_E])

    arb_rows.append({
        'Model': label,
        'delta': f'{delta:.3f}',
        'beta_perm': f'{beta_perm:.3f}' if np.isfinite(beta_perm) else '---',
        'gamma': f'{gamma_med:.3f}' if np.isfinite(gamma_med) else '---',
        'relax_r': f'{relax_med:.3f}' if np.isfinite(relax_med) else '---',
        'A:Concav': 'PASS' if test_A else 'FAIL',
        'B:Perm~1': 'PASS' if test_B else 'FAIL',
        'C:Decay': 'PASS' if test_C else 'FAIL',
        'D:Relax': 'PASS' if test_D else 'FAIL',
        'E:Gather': 'PASS' if test_E else 'FAIL',
        'Score': f'{n_pass}/5',
    })

arb_table = pd.DataFrame(arb_rows)
print('\n── No-Arbitrage Consistency Check ──')
print(arb_table.to_string(index=False))

# LaTeX
print('\n── LaTeX ──')
print('\\begin{tabular}{lccccccccc}')
print('\\toprule')
print('Model & $\\delta$ & $\\beta_p$ & $\\gamma$ & $r$ & A & B & C & D & E \\\\')
print('\\midrule')
for _, row in arb_table.iterrows():
    vals = ' & '.join([row['Model'], row['delta'], row['beta_perm'], row['gamma'],
                       row['relax_r'], row['A:Concav'], row['B:Perm~1'],
                       row['C:Decay'], row['D:Relax'], row['E:Gather']])
    print(f'{vals} \\\\')
print('\\bottomrule')
print('\\end{tabular}')

### Interpretation

- **Test A (Concavity)**: All models should pass since $\beta < 1$ is expected from the square-root law.
- **Test B (Permanent linearity)**: Huberman & Stanzl (2004) prove that permanent impact must be linear in volume to prevent price manipulation. $\beta_{\text{perm}} \approx 1$ is the no-arbitrage requirement.
- **Test C (Decay kernel)**: The power-law decay exponent $\gamma$ should be in a reasonable range. Too small means infinite impact memory; too large means instant reversion.
- **Test D (Relaxation ratio)**: The ratio $r = I_{\text{final}}/I_{\text{peak}}$ should be between 0.5 and 1.0. Values outside this range indicate either excessive reversion (< 0.5) or continued drift (> 1.0).
- **Test E (Gatheral)**: The Gatheral (2010) condition $\delta \leq 1/(1+2\gamma)$ ensures no dynamic arbitrage from round-trip trading. This is the strongest constraint.

---
## 5. Per-Day Beta Analysis (Internal)

Run beta regression separately per day to assess stability of the scaling exponent.

In [ ]:
# ── Section 5: Per-Day Beta ───────────────────────────────────────

perday_results = OrderedDict()

for label, r in R.items():
    pc = r['pc']
    if pc.empty:
        continue

    # Map sample_id to day
    sdm = SAMPLE_DAY_MAP[['sample_id', 'day']].drop_duplicates()
    pc_day = pc.merge(sdm, on='sample_id', how='left')

    day_betas = []
    for day_val, grp in pc_day.groupby('day'):
        bstat = compute_global_beta(grp)
        if np.isfinite(bstat['beta']):
            day_betas.append({'day': day_val, 'beta': bstat['beta'],
                              'r2': bstat['r2'], 'n': bstat['n']})
    day_df = pd.DataFrame(day_betas)
    perday_results[label] = day_df

# ── Table ──
rows = []
for label, ddf in perday_results.items():
    if ddf.empty:
        continue
    b = ddf['beta']
    rows.append({
        'Model': label,
        'Mean beta': f'{b.mean():.3f}',
        'Std': f'{b.std():.3f}',
        'Min': f'{b.min():.3f}',
        'Max': f'{b.max():.3f}',
        'N_days': len(ddf),
    })
table_perday = pd.DataFrame(rows)
print('\n── Per-Day Beta ──')
print(table_perday.to_string(index=False))

In [ ]:
# ── Figure: Per-day beta distributions ──

fig = go.Figure()
for label, ddf in perday_results.items():
    if ddf.empty:
        continue
    fig.add_trace(go.Box(
        y=ddf['beta'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=5, opacity=0.7),
        line_width=1.5))

fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='\u03b2 = 0.5', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=SINGLE_W, height=380, legend_pos='none')
fig.update_xaxes(title_text='')
fig.update_yaxes(title_text='\u03b2 (per day)')
save_fig(fig, '5. Per-Day Beta.png')
fig.show()

---
## Summary

In [ ]:
# ── Grand Summary ─────────────────────────────────────────────────

print('=' * 90)
print(f'{"Model":15s}  {"beta":>6s}  {"beta_rho":>8s}  {"gamma":>6s}  {"b_perm":>6s}  {"b_temp":>6s}  {"Arb":>5s}')
print('-' * 90)
for label in R:
    beta = R[label]['beta']['beta']
    pr = participation_results.get(label, {})
    beta_rho = pr.get('beta_rho', np.nan)
    ddf = decay_results.get(label, pd.DataFrame())
    gamma_med = ddf['gamma'].median() if not ddf.empty else np.nan
    dr = decomp_results.get(label, {})
    bp = dr.get('beta_perm', np.nan)
    bt = dr.get('beta_temp', np.nan)
    arb = [r for r in arb_rows if r['Model'] == label]
    score = arb[0]['Score'] if arb else '---'
    print(f'{label:15s}  {beta:6.3f}  {beta_rho:8.3f}  {gamma_med:6.3f}  {bp:6.3f}  {bt:6.3f}  {score:>5s}')
print('=' * 90)
print(f'{"Theory":15s}  {"0.500":>6s}  {"0.500":>8s}  {"0.5-0.8":>6s}  {"1.000":>6s}  {"0.500":>6s}  {"5/5":>5s}')